# 01 · 数据准备（Colab）

下载语料、建清单、合成固定测试集。

**为什么本地做不了**：C: 盘只剩约 16 GB，而这些语料合计约 23 GB。
详见 `docs/ENVIRONMENT.md`。

## 执行前先看这里

1. **先跑「配置」cell**，确认三件事：
   - `DRIVE_ROOT` 与你实际的 Drive 目录一致（默认 `MyDrive/Audio AI/RTSE`）
   - `DATA_MODE`（默认 `hybrid`：压缩包存 Drive，每会话解压到本地盘）
   - `QUICK_TEST`
2. 代码包 `rtse-colab.zip` 必须已上传到 `DRIVE_ROOT` 下
   （本地执行 `uv run python scripts/pack_for_colab.py` 生成）
3. **第一次强烈建议先把 `QUICK_TEST = True`**，15 分钟验证全流程，再回来跑完整版

## 产出

| 产物 | 位置（都在 `DRIVE_ROOT` 下，持久） | 体积 |
|---|---|---|
| `manifest.json` | 项目根 | 几百 KB |
| `archives/*.tgz` | 项目根（hybrid 模式） | 约 23 GB |
| `testset/` + `testset.zip` | 项目根 | 目标 < 2 GB |

> ⚠️ **测试集必须固定下来**（预先合成好存盘），不能在线随机生成。
> 否则每次评测的噪声段和 SNR 都不同，前后两次跑出来的 CER 没有可比性。
> 训练集则相反 —— 必须在线随机混音，以扩大有效数据量。

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与选择都集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
# 改成你实际存放 rtse-colab.zip 的目录。路径里有空格也没问题。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 原始语料（THCHS-30 / MUSAN / RIRS，合计约 23 GB）怎么放？────────────
#
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每次会话解压到本地盘。
#               一次下载永久有效；训练读取是本地盘全速；
#               每个新会话只需几分钟解压。Drive 占用约 23 GB。
#
#   'local'  —— 全部放本地盘，压缩包用完即删。
#               不占 Drive；代价是**每次新会话都要重下**（15~40 分钟）。
#
#   'drive'  —— 全部放 Drive，直接解压到 Drive。
#               ⚠ 不推荐：THCHS-30 有一万多个小文件，在 Drive 的 FUSE 挂载上
#               逐个创建极慢（每个文件都是一次 API 往返），解压可能要几小时；
#               训练时的随机读取也慢 2~5 倍。只有在 Colab 本地盘不够用时才选它。
DATA_MODE = 'hybrid'

# ── 快速验证模式 ────────────────────────────────────────────────────────
# True  = 只下载 337 MB 的小语料（LibriSpeech dev-clean，英文），
#         约 15 分钟就能把「数据→训练→导出→回传」整条链路跑通一遍。
#         **只用来验证流程，不要用它的结果做最终指标**（英文语料算不了中文 CER）。
# False = 完整流程（THCHS-30 中文 + MUSAN + RIRS，约 23 GB）
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都要靠它

assert DATA_MODE in ('hybrid', 'local', 'drive'), f'DATA_MODE 只能是 hybrid/local/drive'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT

# 压缩包放哪 / 解压到哪 —— 三种模式的唯一区别就在这两行
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{DRIVE}/rawdata' if DATA_MODE == 'drive' else f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'    # local 模式解压后删包省空间，其余保留以便复用

# Drive 侧的产物目录（**这些永远在 Drive 上**，训练结果不能放临时盘）
CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每个 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查两件事：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致'
    '（区分大小写，路径里的空格照写即可）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}' + {
    'hybrid': '   压缩包存 Drive（一次下载永久有效），每会话解压到本地盘',
    'local':  '   全在临时盘，每个新会话都要重新下载',
    'drive':  '   全在 Drive（解压会很慢，一万多个小文件走 FUSE）',
}[DATA_MODE])
print(f'  语料      {"快速验证(小语料/英文)" if QUICK_TEST else "完整流程(中文 THCHS-30)"}')
print()

!df -h /content | tail -1
!df -h /content/drive 2>/dev/null | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 指向的目录。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没有预装的几个包。
# 不用 `pip install -e .`：那会去解析 pyproject 里锁定的 torch CPU 索引，
# 把 Colab 自带的 GPU 版 torch 覆盖掉 —— 训练会瞬间慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。如果 Colab 上的 STFT 与本地哪怕差一点，
# 训练出来的模型拿回本地就会掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

## 1. 下载语料

| 用途 | 数据集 | 体积 | 为什么选它 |
|---|---|---|---|
| 中文语音 | **THCHS-30**（openSLR SLR18） | ~6.4 GB | 带完整转写，可直接算 CER；比 AISHELL-1（15 GB）小得多 |
| 噪声 | **MUSAN** 的 noise 子集（SLR17） | ~11 GB | 真实录制的环境噪声，种类全 |
| 房间冲激响应 | **RIRS_NOISES**（SLR28） | ~6 GB | 含真实测量 + 仿真 RIR，"远场"必需 |
| 冲激型噪声 | **本项目合成** | 0 | 见第 3 节，这一项**不能省** |

**MUSAN 只取 noise 子集**：music 会让模型学会压制音乐（本项目不需要），
speech 子集会与目标语音混淆。

下载支持**断点续传**（`wget -c`）和**镜像回退**。已下好的会跳过 ——
Colab 断线重连后重跑这个 cell 不会重头再来。

In [ ]:
# openSLR 的三个官方镜像。主站在亚洲经常很慢，按顺序回退。

if QUICK_TEST:
    DATASETS = [
        ('librispeech', 12, 'dev-clean.tar.gz', 0.34, 'LibriSpeech'),
    ]
else:
    DATASETS = [
        ('thchs30', 18, 'data_thchs30.tgz',  6.4, 'data_thchs30'),
        ('musan',   17, 'musan.tar.gz',     11.0, 'musan'),
        ('rirs',    28, 'rirs_noises.zip',   6.0, 'RIRS_NOISES'),
    ]

need_gb = sum(d[3] for d in DATASETS)
free_arch = shutil.disk_usage(ARCHIVE_DIR).free / 1e9
free_data = shutil.disk_usage(DATA).free / 1e9
print(f'需要约 {need_gb:.1f} GB')
print(f'  压缩包卷 {ARCHIVE_DIR}  可用 {free_arch:.1f} GB')
print(f'  解压卷   {DATA}  可用 {free_data:.1f} GB')
# hybrid/local 下压缩包与解压结果在不同卷上，各自只需 1 倍多一点
assert free_arch > need_gb * 1.15, '压缩包卷空间不够'
assert free_data > need_gb * 1.15, (
    '解压卷空间不够。Colab 本地盘通常有 100+ GB；若确实不够，'
    '先设 QUICK_TEST=True 跑小语料验证流程。'
)

In [ ]:
MIRRORS = [
    'https://www.openslr.org/resources',
    'https://openslr.elda.org/resources',          # 欧洲
    'https://openslr.magicdatatech.com/resources', # 中国
]

def fetch(name, slr, fname, expect_dir):
    """下载 → 解压 → 校验。

    **两个独立的完成标记**，这是 hybrid 模式能省下重复下载的关键：
      - `{ARCHIVE_DIR}/.{name}.downloaded` —— 压缩包已下好（在 Drive 上，跨会话保留）
      - `{DATA}/.{name}.extracted`         —— 已解压到当前卷（本地盘，会话内有效）

    新会话里第一个标记还在、第二个没了 → 只解压，不重下。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    archive = f'{ARCHIVE_DIR}/{fname}'

    if os.path.exists(ex_mark) and os.path.isdir(f'{DATA}/{expect_dir}'):
        print(f'[skip  ] {name} 已解压'); return True

    # ── 下载（已有则跳过）──────────────────────────────────────────────
    if os.path.exists(dl_mark) and os.path.exists(archive):
        print(f'[cached] {name} 压缩包已在 Drive ({os.path.getsize(archive)/1e9:.2f} GB)，跳过下载')
    else:
        ok = False
        for base in MIRRORS:
            url = f'{base}/{slr}/{fname}'
            print(f'[get   ] {name} ← {url}')
            # -c 断点续传：断线后重跑不会从头下
            rc = os.system(f'wget -q --show-progress -c -T 30 -O {shq(archive)} {shq(url)}')
            if rc == 0 and os.path.exists(archive) and os.path.getsize(archive) > 1e6:
                ok = True
                break
            print('[retry ] 该镜像失败，换下一个')
        if not ok:
            print(f'[FAIL  ] {name} 三个镜像都下不下来。'
                  f'去 https://www.openslr.org/{slr}/ 确认文件名是否变了。')
            return False
        Path(dl_mark).touch()

    # ── 解压 ───────────────────────────────────────────────────────────
    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {DATA}')
    t = time.time()
    if fname.endswith('.zip'):
        rc = os.system(f'unzip -q -o {shq(archive)} -d {shq(DATA)}')
    else:
        rc = os.system(f'tar -xzf {shq(archive)} -C {shq(DATA)}')

    # 解压后必须校验目录真的存在：压缩包被截断时 tar 会部分成功，
    # 目录建出来了但内容不全 —— 不校验的话要等到后面以"扫到 0 个文件"的形式暴露。
    if not os.path.isdir(f'{DATA}/{expect_dir}'):
        print(f'[FAIL  ] 解压后没有找到 {DATA}/{expect_dir}（rc={rc}）')
        print(f'         压缩包可能不完整，删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive)
        Path(dl_mark).unlink(missing_ok=True)
    Path(ex_mark).touch()
    print(f'[done  ] {name}   解压耗时 {time.time()-t:.0f} 秒')
    return True

t0 = time.time()
results = {n: fetch(n, s, f, d) for n, s, f, _, d in DATASETS}
print(f'\n总耗时 {(time.time()-t0)/60:.1f} 分钟   结果: {results}')
assert all(results.values()), '有数据集没就绪，看上面的 FAIL 信息'

!df -h {shq(DATA)} | tail -1
!ls -1 {shq(DATA)}

## 2. 建立文件清单（按说话人划分）

In [ ]:
import random

def scan(root, exts=('.wav', '.flac')):
    root = Path(root)
    if not root.exists(): return []
    return sorted(str(p) for p in root.rglob('*') if p.suffix.lower() in exts)

if QUICK_TEST:
    speech_all = scan(f'{DATA}/LibriSpeech')
    noise_all = []          # 快速模式不下 MUSAN，只用合成噪声
    rir_all = []            # 也不下 RIRS，用合成 RIR
else:
    speech_all = scan(f'{DATA}/data_thchs30/data')
    # 只取 noise 子集，不要 music / speech（理由见上文）
    noise_all = scan(f'{DATA}/musan/noise')
    rir_all = (scan(f'{DATA}/RIRS_NOISES/simulated_rirs')
               + scan(f'{DATA}/RIRS_NOISES/real_rirs_isotropic_noises'))

print(f'语音 {len(speech_all):>6} 条')
print(f'噪声 {len(noise_all):>6} 条' + ('   (快速模式：仅用合成噪声)' if QUICK_TEST else ''))
print(f'RIR  {len(rir_all):>6} 条' + ('   (快速模式：仅用合成 RIR)' if QUICK_TEST else ''))
assert speech_all, '没扫到语音文件，检查上一步的解压结果'

In [ ]:
# 按**说话人**划分，不是按文件随机划分。
# 按文件随机划分会让同一个说话人同时出现在训练集和测试集里，
# 模型可以靠"记住这个人的音色"作弊，测出来的指标会虚高。
def speaker_of(p):
    stem = Path(p).stem
    return stem.split('_')[0] if '_' in stem else stem.split('-')[0]

spk = sorted({speaker_of(p) for p in speech_all})
random.Random(20260805).shuffle(spk)
n_test = max(2, len(spk) // 10)
n_val = max(2, len(spk) // 10)
spk_test = set(spk[:n_test])
spk_val = set(spk[n_test:n_test + n_val])
spk_train = set(spk[n_test + n_val:])
print(f'说话人 {len(spk)} 位 → train {len(spk_train)} / val {len(spk_val)} / test {len(spk_test)}')

split = {'train': [], 'val': [], 'test': []}
for p in speech_all:
    s = speaker_of(p)
    split['test' if s in spk_test else 'val' if s in spk_val else 'train'].append(p)

# 噪声与 RIR 同样要划分：测试用的必须是训练没见过的
rnd = random.Random(42)
nz = noise_all[:]; rr = rir_all[:]
rnd.shuffle(nz); rnd.shuffle(rr)
nz_test, nz_train = nz[:len(nz)//5], nz[len(nz)//5:]
rir_test, rir_train = rr[:len(rr)//5], rr[len(rr)//5:]

for k, v in split.items():
    print(f'  语音 {k:>5}: {len(v):>6} 条')
print(f'  噪声 train/test: {len(nz_train)}/{len(nz_test)}')
print(f'  RIR  train/test: {len(rir_train)}/{len(rir_test)}')

manifest = {
    'quick_test': QUICK_TEST,
    'data_dir': DATA,
    'speech': split,
    'noise_train': nz_train, 'noise_test': nz_test,
    'rir_train': rir_train, 'rir_test': rir_test,
    'speaker_split': {'train': sorted(spk_train), 'val': sorted(spk_val), 'test': sorted(spk_test)},
}

## 3. 补充冲激型噪声

**这一步不能省。** 本地实验（`docs/FINDINGS.md` F-02）已经证明：
键盘敲击这类冲激噪声会让**所有** MCRA 类 DSP 方法失效（ΔSI-SDR 为负，
谱减法甚至掉 0.7 dB）。根因是 MCRA 假设"噪声是谱的下包络、语音是其上的瞬态"，
而冲激噪声在它看来和语音起始一模一样，于是噪声估计**根本不更新**。

这正是神经网络最有说服力的立足点 —— 但前提是训练集里得有这类噪声。
MUSAN 里这类样本偏少，所以额外合成一批混进去。

In [ ]:
import numpy as np, soundfile as sf
from rtse.data.synth import make_noise

IMPULSIVE = f'{DATA}/synth_impulsive'
os.makedirs(IMPULSIVE, exist_ok=True)
rng = np.random.default_rng(7)

made = []
for kind in ['keyboard', 'cafeteria', 'hum', 'white', 'pink', 'car', 'babble']:
    for i in range(30):                       # 每类 30 条 × 10 秒
        y = make_noise(kind, 16000 * 10, rng)
        y = y / (np.max(np.abs(y)) + 1e-9) * 0.7
        fn = f'{IMPULSIVE}/{kind}_{i:03d}.wav'
        sf.write(fn, y, 16000, subtype='PCM_16')
        made.append(fn)

rnd.shuffle(made)
cut = len(made) // 5
manifest['noise_test'] += made[:cut]
manifest['noise_train'] += made[cut:]

Path(f'{DRIVE}/manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False), encoding='utf-8')
print(f'合成 {len(made)} 条噪声 → 训练 +{len(made)-cut} / 测试 +{cut}')
print(f'清单已写入 {DRIVE}/manifest.json')
print(f'  训练噪声总数 {len(manifest["noise_train"])}，测试噪声总数 {len(manifest["noise_test"])}')

## 4. 合成固定测试集

**分层设计**，让每个维度都能单独扫描：

- **主表**（SNR × 噪声类型）：固定 T60 = 0.3 s
- **混响扫描**：固定 SNR = 5 dB、noise = babble，扫 T60

体积按本地 C: 盘的预算（< 2 GB）定，生成后会打印实际体积。

**⚠️ 音频长度按每条语音的自然时长决定，不再硬截成固定秒数**——早期版本
把每条语音硬截到固定 6 秒，但 THCHS-30 的句子长度参差不齐（不少要 7~10 秒
才能读完），硬截会把长句子从中间切断，**可参考转写文本（`text` 字段）
用的是完整原句**，于是"识别没说完的话"被记成"识别错误"，CER 系统性偏高，
而且跟降噪质量毫无关系——这是真实跑通 ASR 之后才用 VAD 查出来的
（见 `docs/ISSUES.md` I-21）。改成按自然时长保留完整内容，只对下限/上限做保护。

**⚠️ 截断校验改成了结构性断言，不再用 VAD 判断"是否还在说话"**——那条启发式
先后被两种声学现象污染过：混响拖尾（第一次踩坑）、真实语音结尾处 VAD 的
自然 hangover（第二次踩坑，实测 780 条里 477 条命中，即便已经改成查混响前的
干信号）。真正该验证的其实是一个纯代码逻辑的不变量：走到写入这一步时，
前面的重试循环已经保证 `duration <= MAX_SEG_SEC`，所以 `seg_len >= clean.size`
必然成立，`clean[:seg_len]` 对内容一定是无操作——这是数学上能证明的，不需要
再去猜"多长的静音才算正常"。改成直接断言这个不变量，不再依赖任何声学假设。

> 想先跑通流程再生成完整 780 条：把下面 `PER_CELL` 从 `20` 改成 `2`
> 跑一轮（39 格 × 2 ≈ 80 个样本，几十秒能跑完），确认没有报错再改回 `20`。

In [ ]:
from rtse.audio.io import read_audio, write_audio
from rtse.data.synth import mix_at_snr, apply_rir, make_rir
from tqdm.auto import tqdm

SNRS = [-5, 0, 5, 10, 15, 20]
NOISES = ['keyboard', 'cafeteria', 'babble', 'white', 'car', 'hum']  # 合成噪声，类型明确可控
T60S = [0.0, 0.3, 0.6, 0.9]
PER_CELL = 20
MIN_SEG_SEC, MAX_SEG_SEC = 3.0, 18.0  # 自然时长的下限/上限保护，见上方说明

def read_transcript(wav_path):
    """读 THCHS-30 的转写（.wav.trn）。快速模式的 LibriSpeech 没有，返回 None。"""
    trn = Path(str(wav_path) + '.trn')
    if not trn.exists():
        return None
    lines = trn.read_text(encoding='utf-8').strip().splitlines()
    if not lines:
        return None
    first = lines[0].strip()
    if first.endswith('.trn'):                 # 部分是软链接，指向真正的 trn
        real = Path(wav_path).parent.parent / first
        if real.exists():
            lines = real.read_text(encoding='utf-8').strip().splitlines()
    return lines[0].replace(' ', '') if lines else None

test_speech = [p for p in split['test'] if read_transcript(p)] or split['test']
has_text = read_transcript(test_speech[0]) is not None
print(f'测试集可用语音 {len(test_speech)} 条，带转写: {has_text}')
if not has_text:
    print('⚠ 没有转写文本 → 后续算不了 CER。快速模式下属正常。')

cells = [{'snr': s, 'noise': n, 't60': 0.3} for s in SNRS for n in NOISES]
cells += [{'snr': 5, 'noise': 'babble', 't60': t} for t in T60S if t != 0.3]
n_samples = len(cells) * PER_CELL
print(f'{len(cells)} 格 × {PER_CELL} 条 = {n_samples} 个样本（体积取决于实际语音时长，生成后用 du -sh 看）')

In [ ]:
os.makedirs(f'{TESTSET_DIR}/audio', exist_ok=True)
rng = np.random.default_rng(20260805)
records, sidx = [], 0
skipped_too_long = 0

for ci, cell in enumerate(tqdm(cells, desc='合成测试集')):
    for k in range(PER_CELL):
        # 固定的取样顺序 → 整个测试集完全可复现。如果自然时长超过 MAX_SEG_SEC
        # （极少数长句），依次往后找下一条，不静默截断导致文本对不上音频。
        idx0 = ci * PER_CELL + k
        for attempt in range(len(test_speech)):
            sp = test_speech[(idx0 + attempt) % len(test_speech)]
            clean = read_audio(sp)
            if clean.size / 16000 <= MAX_SEG_SEC:
                break
        else:
            continue  # 理论上不会发生，整个语料都超长的话原本就该换语料
        if clean.size / 16000 > MAX_SEG_SEC:
            skipped_too_long += 1

        seg_len = int(np.clip(clean.size / 16000, MIN_SEG_SEC, MAX_SEG_SEC) * 16000)
        # 结构性校验，取代不可靠的 VAD 启发式（见上方说明）：走到这里
        # duration<=MAX_SEG_SEC 已经成立，所以 seg_len 一定 >= clean.size。
        assert clean.size <= seg_len, f'{sp}: seg_len 比原始语音还短，逻辑错误，不应该发生'
        if clean.size < seg_len:
            clean = np.pad(clean, (0, seg_len - clean.size))
        clean = clean[:seg_len]
        clean = clean / (np.max(np.abs(clean)) + 1e-9) * 0.7

        wet = apply_rir(clean, make_rir(cell['t60'], rng=rng)) if cell['t60'] > 0 else clean
        noisy, _ = mix_at_snr(wet, make_noise(cell['noise'], seg_len, rng), cell['snr'], rng=rng)

        stem = f"{sidx:05d}_{cell['noise']}_snr{cell['snr']}_t{cell['t60']}"
        write_audio(f'{TESTSET_DIR}/audio/{stem}_noisy.wav', noisy)
        # 参考 = **混响后**的干净语音，不是原始干信号。
        # 否则降噪模型会因为"没能去掉混响"被扣分，混淆了降噪与去混响两件事。
        write_audio(f'{TESTSET_DIR}/audio/{stem}_clean.wav', wet)
        records.append({'id': stem, 'noisy': f'audio/{stem}_noisy.wav',
                        'clean': f'audio/{stem}_clean.wav', 'duration_s': round(seg_len/16000, 2),
                        'text': read_transcript(sp), 'source': str(sp), **cell})
        sidx += 1

print(f'跳过的超长句子（>{MAX_SEG_SEC}s，理论上不该发生）: {skipped_too_long}')
print(f'结构性校验全部通过（{len(records)} 条）：内部断言没有触发，'
      f'说明每条参考音频都完整保留了原始语音内容，没有被截断。')
Path(f'{TESTSET_DIR}/index.json').write_text(
    json.dumps({'sample_rate': 16000, 'min_seg_sec': MIN_SEG_SEC, 'max_seg_sec': MAX_SEG_SEC,
                'per_cell': PER_CELL, 'quick_test': QUICK_TEST, 'records': records},
               ensure_ascii=False, indent=1), encoding='utf-8')
print(f'测试集 {len(records)} 个样本 → {TESTSET_DIR}')
!du -sh "{TESTSET_DIR}"

## 5. 打包测试集，下载到本地

In [ ]:
!cd "{DRIVE}" && rm -f testset.zip && zip -q -r testset.zip testset && ls -lh testset.zip
print()
print('下一步：')
print('  1. 继续跑 02_train.ipynb（数据清单已就绪）')
print(f'  2. 有空时从 Drive 下载 {DRIVE}/testset.zip，解压到本地项目的 data/ 下，')
print('     使 data/testset/index.json 存在，然后本地 `uv run rtse-doctor` 会变绿')